# Лабораторна робота № 10
## Побудова нейронної мережі засобами Keras. Налаштування гіперпараметрів

**Дисципліна:** Машинне навчання  
**Прізвище, ім’я:** Соловей  
**Група:** 3-12  
**Дата:** ______________________________

**Мета:** навчитися завантажувати та готувати дані Fashion MNIST, створювати і навчати нейронну мережу засобами Keras, оцінювати якість класифікації, а також підбирати гіперпараметри за допомогою Keras Tuner.

**Порядок виконання:** послідовно виконайте кожен пункт у наведених комірках. Доповніть код, збережіть графіки та результати виконання, сформулюйте власні висновки.

### Вхідні дані та критерії

Використовуйте **Fashion MNIST** — зображення розміром **28 × 28 пікселів** із **10 класів**.  
У **завданні 1** цільова точність на тестовій вибірці — **не нижче 0,87** (відповідно до вихідної лабораторної роботи).

**Важливо:** тестова вибірка призначена для фінального оцінювання. Для навчання, ранньої зупинки та добору гіперпараметрів використовуйте окрему валідаційну частину навчальних даних; не налаштовуйте модель за результатами тестової вибірки.

## Підготовка робочого середовища

Імпортуйте необхідні бібліотеки. У другому завданні буде потрібно встановити та підключити **Keras Tuner**.

In [ ]:
# Підготуйте імпорти TensorFlow / Keras, NumPy, Matplotlib,
# classification_report, а також потрібних callback-функцій.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from IPython.display import display, Markdown
keras.utils.set_random_seed(42)

# Завдання 1. Класифікація Fashion MNIST за допомогою Keras

### Крок 1. Завантаження даних

Завантажте Fashion MNIST. Створіть навчальну й тестову вибірки та перелік назв **10 класів**. Виведіть розміри масивів даних.

In [ ]:
# Завантажте Fashion MNIST, визначте class_names.
# Виведіть форми x_train, y_train, x_test, y_test.
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
class_names = ['Футболка', 'Штани', 'Пуловер', 'Сукня', 'Пальто', 'Сандалі', 'Сорочка', 'Кросівки', 'Сумка', 'Черевики']
for name, arr in [('x_train', x_train), ('y_train', y_train), ('x_test', x_test), ('y_test', y_test)]:
    print(name, arr.shape)

### Крок 2. Візуальний аналіз

Виведіть **10 зображень** навчальної вибірки. Для кожного підпишіть відповідний клас.

In [ ]:
# Побудуйте візуалізацію 10 зображень із підписами класів.
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.tight_layout(); plt.show()

### Крок 3. Нормалізація

Нормалізуйте значення пікселів до діапазону **[0; 1]**. Переконайтеся, що тип даних придатний для навчання нейронної мережі.

In [ ]:
# Нормалізуйте навчальні й тестові дані.
# Перевірте діапазони значень.
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
print(x_train.dtype, x_train.min(), x_train.max())
print(x_test.dtype, x_test.min(), x_test.max())
x_fit, x_val, y_fit, y_val = train_test_split(x_train, y_train, test_size=0.15, random_state=42, stratify=y_train)

### Крок 4. Побудова й компіляція моделі

Створіть послідовну нейронну мережу. Використайте перетворення зображення у вектор, повнозв’язні шари з **ReLU**, регуляризацію **Dropout** та вихід із **10 нейронів із softmax**. Налаштуйте оптимізатор Adam, функцію втрат для багатокласової класифікації з цілочисельними мітками та метрику accuracy.

Поясніть вибір кількості нейронів, dropout і швидкості навчання.

In [ ]:
# Створіть і скомпілюйте модель.
model = keras.Sequential([
    keras.Input(shape=(28, 28)), layers.Flatten(),
    layers.Dense(512, activation='relu'), layers.Dropout(0.2),
    layers.Dense(256, activation='relu'), layers.Dropout(0.2),
    layers.Dense(128, activation='relu'), layers.Dropout(0.1),
    layers.Dense(10, activation='softmax')])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# 512, 256 і 128 нейронів поступово стискають представлення ознак.
# Dropout зменшує перенавчання; 0.001 — стандартний початковий крок Adam.

### Крок 5. Навчання моделі

Навчіть модель із виділенням **валідаційної частини з навчальної вибірки**. Застосуйте **EarlyStopping** і **ReduceLROnPlateau**. Збережіть історію навчання для подальших графіків.

In [ ]:
# Налаштуйте callbacks і навчіть модель.
# Збережіть history.
def callbacks():
    return [EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True), ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)]
history = model.fit(x_fit, y_fit, validation_data=(x_val, y_val), epochs=50, batch_size=128, callbacks=callbacks(), verbose=2)

### Крок 6. Аналіз архітектури

Виведіть підсумкову структуру нейронної мережі, кількість шарів і параметрів.

In [ ]:
# Виведіть підсумок архітектури моделі.
model.summary()
print('Кількість шарів:', len(model.layers))
print('Кількість параметрів:', model.count_params())

### Крок 7. Збереження моделі

Збережіть навчену модель у форматі **`.keras`**. Під час роботи в Google Colab за потреби підключіть Google Drive.

In [ ]:
# Збережіть навчену модель (.keras).
# За потреби підключіть Google Drive.
model.save('fashion_mnist_base.keras')
print('Модель збережено у fashion_mnist_base.keras')

### Крок 8. Графіки навчання

Побудуйте два графіки: **функція втрат (loss)** і **точність (accuracy)** для навчальної та валідаційної вибірок. Підпишіть осі, легенди й назви графіків.

In [ ]:
# Побудуйте графіки loss/val_loss та accuracy/val_accuracy.
def plot_history(hist):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, title in zip(axes, ['loss', 'accuracy'], ['Функція втрат', 'Точність']):
        epochs = range(1, len(hist.history[metric])+1)
        ax.plot(epochs, hist.history[metric], label='Навчання')
        ax.plot(epochs, hist.history['val_'+metric], label='Валідація')
        ax.set(xlabel='Епоха', ylabel=metric, title=title)
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
plot_history(history)

### Крок 9. Оцінювання моделі

Один раз оцініть навчену модель на **тестовій вибірці**. Отримайте прогнози й сформуйте **класифікаційний звіт** (precision, recall, F1-score) для всіх класів.

Порівняйте отриману accuracy з порогом **0,87**.

In [ ]:
# Обчисліть loss і accuracy на тестовій вибірці.
# Знайдіть прогнозовані класи та сформуйте classification_report.
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
y_pred = model.predict(x_test, verbose=0).argmax(axis=1)
print(f'Loss: {test_loss:.4f}; Accuracy: {test_acc:.4f}')
print('Поріг 0.87:', 'досягнуто' if test_acc >= 0.87 else 'не досягнуто')
print(classification_report(y_test, y_pred, target_names=class_names, digits=4, zero_division=0))
cm = confusion_matrix(y_test, y_pred)
errors = cm.sum(axis=1) - cm.diagonal()
worst = np.argsort(errors)[-3:][::-1]

### Крок 10. Візуалізація прогнозів

Покажіть **10 тестових зображень**. Для кожного вкажіть правильний і прогнозований клас, а правильність прогнозу позначте кольором підпису.

In [ ]:
# Візуалізуйте 10 зображень: справжні й прогнозовані класи.
plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_test[i], cmap='gray')
    plt.title(f'Факт: {class_names[y_test[i]]}\nПрогноз: {class_names[y_pred[i]]}', color='green' if y_test[i] == y_pred[i] else 'red', fontsize=10)
    plt.axis('off')
plt.tight_layout(); plt.show()

**Проміжний висновок до завдання 1**

Запишіть значення тестової accuracy, охарактеризуйте класи з найбільшою кількістю помилок та поясніть, чи виконано вимогу щодо мінімальної точності.

**Ваш висновок:**  
Результат і аналіз помилок формуються автоматично після виконання наступної комірки.

In [ ]:
display(Markdown(f"Тестова accuracy: **{test_acc:.4f}**. Вимогу 0,87 {'виконано' if test_acc >= .87 else 'не виконано'}. Найбільше помилок: " + ', '.join(f'{class_names[i]} ({errors[i]})' for i in worst) + '. Схожі за формою предмети одягу важче розрізняти.'))

# Завдання 2. Добір гіперпараметрів за допомогою Keras Tuner

На основі того самого набору даних виконайте автоматизований пошук конфігурації нейронної мережі. У вихідній лабораторній роботі використовується **RandomSearch**.

### Крок 1. Підготовка Keras Tuner

Установіть `keras-tuner` (за потреби) й імпортуйте засоби для пошуку гіперпараметрів, шари Dense, Flatten, Dropout та регуляризацію L2.

In [ ]:
# За потреби встановіть keras-tuner.
# Імпортуйте RandomSearch та потрібні компоненти Keras.
import sys, subprocess, importlib.util
if importlib.util.find_spec('keras_tuner') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'keras-tuner'])
from keras_tuner import RandomSearch

### Крок 2. Підготовка набору даних

Повторно використайте Fashion MNIST у нормалізованому вигляді. Переконайтеся, що вибірка для **валідації відокремлена від тестової**. Не передавайте `x_test`, `y_test` у процес пошуку гіперпараметрів.

In [ ]:
# Підготуйте дані для добору гіперпараметрів і фінального тестування.
print('Навчальна:', x_fit.shape, y_fit.shape)
print('Валідаційна:', x_val.shape, y_val.shape)
print('Тестова:', x_test.shape, y_test.shape)

### Крок 3. Функція побудови моделі

Реалізуйте функцію, яка приймає набір гіперпараметрів та повертає скомпільовану модель. Передбачте пошук **кількості прихованих шарів (1–3)**, **кількості нейронів (32–512 із кроком 32)** та **швидкості навчання (0,01; 0,001; 0,0001)**. Використайте ReLU, Dropout, L2 та softmax, як у вихідному варіанті лабораторної роботи.

In [ ]:
# Опишіть функцію build_model(hp).
# Задайте області пошуку гіперпараметрів.
def build_model(hp):
    m = keras.Sequential([keras.Input(shape=(28, 28)), layers.Flatten()])
    depth = hp.Int('num_layers', 1, 3)
    for i in range(3):
        with hp.conditional_scope('num_layers', list(range(i+1, 4))):
            units = hp.Int(f'units_{i}', 32, 512, step=32)
            if i < depth:
                m.add(layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(1e-5)))
                m.add(layers.Dropout(0.2))
    m.add(layers.Dense(10, activation='softmax'))
    lr = hp.Choice('learning_rate', [0.01, 0.001, 0.0001])
    m.compile(optimizer=keras.optimizers.Adam(lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

### Крок 4. Налаштування пошуку

Створіть **RandomSearch** з критерієм **val_accuracy**. У вихідному прикладі передбачено **3 спроби (max_trials = 3)** і **1 виконання на спробу**. Зазначте каталог збереження результатів пошуку.

In [ ]:
# Ініціалізуйте RandomSearch.
tuner = RandomSearch(build_model, objective='val_accuracy', max_trials=3, executions_per_trial=1, directory='keras_tuner_results', project_name='fashion_mnist', overwrite=True, seed=42)
tuner.search_space_summary()

### Крок 5. Пошук та вибір найкращої конфігурації

Запустіть пошук із **EarlyStopping** і **ReduceLROnPlateau**, використовуючи лише навчальні та валідаційні дані. Виведіть знайдені гіперпараметри та структуру найкращої моделі. За потреби перенавчіть обрану конфігурацію, зберігши історію навчання.

In [ ]:
# Виконайте пошук, отримайте найкращі гіперпараметри і модель.
# Навчіть вибрану модель без використання тестових даних.
tuner.search(x_fit, y_fit, validation_data=(x_val, y_val), epochs=30, batch_size=128, callbacks=callbacks(), verbose=2)
best_hp = tuner.get_best_hyperparameters(1)[0]
print('Найкращі гіперпараметри:', best_hp.values)
best_model = tuner.hypermodel.build(best_hp)
best_history = best_model.fit(x_fit, y_fit, validation_data=(x_val, y_val), epochs=50, batch_size=128, callbacks=callbacks(), verbose=2)
best_model.summary()

### Крок 6. Фінальне оцінювання та графіки

Оцініть найкращу модель на **тестовій вибірці**. Наведіть тестові loss і accuracy. Побудуйте графіки навчальної/валідаційної функції втрат і точності.

In [ ]:
# Оцініть найкращу модель та побудуйте графіки навчання.
best_loss, best_acc = best_model.evaluate(x_test, y_test, verbose=0)
print(f'Loss: {best_loss:.4f}; Accuracy: {best_acc:.4f}')
plot_history(best_history)
best_model.save('fashion_mnist_tuned.keras')

# Висновки. Порівняння моделей

Порівняйте модель із завдання 1 та модель після автоматизованого налаштування. Результати внесіть до таблиці.

**Таблиця та відповіді формуються автоматично в наступній комірці за фактичними результатами навчання.**

In [ ]:
depth = best_hp.get('num_layers')
units = [best_hp.get(f'units_{i}') for i in range(depth)]
comparison = pd.DataFrame({'Показник': ['Кількість прихованих шарів', 'Кількість нейронів', 'Початкова швидкість навчання', 'Accuracy', 'Loss', 'Кількість епох'], 'Модель 1': [3, '512, 256, 128', 0.001, round(test_acc, 4), round(test_loss, 4), len(history.epoch)], 'Keras Tuner': [depth, str(units), best_hp.get('learning_rate'), round(best_acc, 4), round(best_loss, 4), len(best_history.epoch)]})
display(comparison)
change = best_acc - test_acc
hard = ', '.join(f'{class_names[i]} ({errors[i]} помилок)' for i in worst)
display(Markdown(f"**Висновки.** У завданні 1 точність становить {test_acc:.4f}; поріг 0,87 {'досягнуто' if test_acc >= .87 else 'не досягнуто'}. Після оптимізації точність становить {best_acc:.4f}, зміна — {change*100:+.2f} відсоткового пункта. Найкраща за валідацією конфігурація серед трьох спроб: {depth} прихованих шарів, нейрони {units}, початкова швидкість навчання {best_hp.get('learning_rate')}. Результати можуть відрізнятися через випадкову ініціалізацію, порядок навчання та особливості обчислень. Найскладніші класи базової моделі: {hard}. Подібні фасони одягу можуть мати схожі контури на зображеннях 28 × 28."))

---
**Перед поданням перевірте:** заповнені всі комірки; результати виконання збережені; є графіки, класифікаційний звіт, результати двох моделей і власні висновки. Шаблон не містить готових розв’язків.